# Notebook 1 - Data Preprocessing for the Econ Games practice round

This notebook is **only for preprocessing**. The goal here is to make the three source tables clean, consistent, and easy to use in later notebooks **without** doing feature engineering or building the modeling dataset yet.

We will keep the three tables separate:

- **`laurel_gate_clean`**: the current Laurel Park GPS races, still at the gate level
- **`laurel_horse_race_flat`**: the Laurel table flattened to one row per horse-race, with raw gate columns widened out
- **`pps_clean`**: the traditional past-performance table
- **`gps_pp_clean`**: the historical GPS past-performance table

We will also create a **distance-aware point-of-call reference table** so the next notebook knows which point-of-call columns are actually used at each race distance.

**What this notebook does**

1. Load the workbook and inspect the three relevant tabs  
2. Standardize column names and clean whitespace / blank strings  
3. Parse dates and numeric columns carefully  
4. Handle **structural missingness** correctly  
   - blank `about_distance_indicator` means "not about"
   - blank `grade` usually means "not a graded stakes race"
   - blank GPS flag in the PP table means "no GPS available for that PP"
5. Normalize race-distance fields so every table has:
   - `distance_furlongs`
   - `distance_meters_expected`
   - `distance_label`
6. Clean the point-of-call columns in the PP table
7. Keep the historical tables separate, but add reliable keys so they can be connected later
8. Flatten the Laurel gate-level table into a wide horse-race table **without engineering features**
9. Save clean outputs for Notebook 2

## 0. Notebook Setup

In [210]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

In [211]:
# Paths and constants

WORKBOOK_NAME = "GPS Races for LRL Practice Round.xlsx"
FURLONG_TO_METERS = 201.168

candidate_paths = [
    Path(WORKBOOK_NAME),
    Path("../data/raw") / WORKBOOK_NAME,
]

RAW_XLSX_PATH = None
for p in candidate_paths:
    if p.exists():
        RAW_XLSX_PATH = p
        break

if RAW_XLSX_PATH is None:
    raise FileNotFoundError(
        f"Could not find {WORKBOOK_NAME}. Put it next to this notebook or update RAW_XLSX_PATH."
    )

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_XLSX_PATH, OUTPUT_DIR.resolve()

(PosixPath('../data/raw/GPS Races for LRL Practice Round.xlsx'),
 PosixPath('/Users/amourtu1934/Documents/4. Personal Projects/horse-racing-prediction/practice_round/data/processed'))

## 1. Load Data

The given workbook contains four tabs in total:

1. **Laurel Park GPS races (past 60d)** -> current target races, gate-level GPS
2. **Starters pps (relating to Laurel)** -> traditional historical PP rows
3. **GPS PPS** -> historical PP races that also have GPS, again gate-level
4. **notes** -> a small note tab that is not central to our later work

We will work and clean the first three tabs. Also, it is indeed very helpful to read the README file to understand the columns meanings beforehand.

In [212]:
workbook = pd.ExcelFile(RAW_XLSX_PATH)

sheet_inventory = pd.DataFrame({
    "sheet_name": workbook.sheet_names,
})

sheet_inventory["n_rows"] = [
    workbook.parse(name).shape[0] for name in workbook.sheet_names
]
sheet_inventory["n_cols"] = [
    workbook.parse(name).shape[1] for name in workbook.sheet_names
]

display(sheet_inventory)


,sheet_name,n_rows,n_cols
0,Laurel Park GPS races (past 60d,14642,25
1,Starters pps (relating to Laure,2897,36
2,GPS PPS,31174,26
3,notes,1,1


In [213]:
# Read the three main tabs
laurel_raw = workbook.parse("Laurel Park GPS races (past 60d")
pps_raw    = workbook.parse("Starters pps (relating to Laure")
gps_pp_raw = workbook.parse("GPS PPS")

print("Laurel raw shape:", laurel_raw.shape)
print("Traditional PP raw shape:", pps_raw.shape)
print("GPS PP raw shape:", gps_pp_raw.shape)

Laurel raw shape: (14642, 25)
Traditional PP raw shape: (2897, 36)
GPS PP raw shape: (31174, 26)


### a. The Laurel Park GPS Races Dataset

In [214]:
# First 5 rows of laurel_raw
laurel_raw.head()

,track_id,race_date,race_number,distance_id,distance_unit,about_distance_indicator,Distance,surface,race_type,grade,Purse,registration_number,horse_name,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides
0,LRL,2025-12-12,1,7.0,F,,7F,D,SOC,,27900,19002714,Mischief Motion,4,0.0,7,7,7.45,90.517,2.057,27.5,100.6,1417.6,16.6,216.0
1,LRL,2025-12-12,1,7.0,F,,7F,D,SOC,,27900,19002714,Mischief Motion,4,0.5,7,7,7.13,83.068,1.424,20.2,100.6,1317.0,16.4,199.4
2,LRL,2025-12-12,1,7.0,F,,7F,D,SOC,,27900,19002714,Mischief Motion,4,1.0,7,7,7.06,75.939,1.269,18.2,100.6,1216.4,16.4,183.0
3,LRL,2025-12-12,1,7.0,F,,7F,D,SOC,,27900,19002714,Mischief Motion,4,1.5,6,7,7.32,68.874,1.250,18.3,103.8,1115.8,16.8,166.6
4,LRL,2025-12-12,1,7.0,F,,7F,D,SOC,,27900,19002714,Mischief Motion,4,2.0,5,7,6.98,61.555,1.098,16.3,104.6,1012.0,16.4,149.8


In [215]:
# Basic information of columns in laurel_raw
laurel_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14642 entries, 0 to 14641
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   track_id                  14642 non-null  object        
 1   race_date                 14642 non-null  datetime64[ns]
 2   race_number               14642 non-null  int64         
 3   distance_id               14642 non-null  float64       
 4   distance_unit             14642 non-null  object        
 5   about_distance_indicator  14642 non-null  object        
 6   Distance                  14642 non-null  object        
 7   surface                   14642 non-null  object        
 8   race_type                 14642 non-null  object        
 9   grade                     14642 non-null  object        
 10  Purse                     14642 non-null  int64         
 11  registration_number       14642 non-null  object        
 12  horse_name        

In [216]:
# Basic statistical summary
laurel_raw.describe()

,race_date,race_number,distance_id,Purse,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides
count,14642,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000,14642.000000
mean,2026-01-04 20:45:10.449392128,5.146428,7.140008,38664.161317,4.248737,3.319082,4.224423,4.224423,6.353055,47.370359,0.723426,11.149597,101.038553,771.427380,14.518713,109.736696
min,2025-12-12 00:00:00,1.000000,5.500000,18410.000000,1.000000,0.000000,1.000000,1.000000,3.460000,6.115000,0.000000,0.000000,100.500000,100.500000,0.000000,0.000000
25%,2025-12-20 00:00:00,3.000000,6.000000,24550.000000,2.000000,1.500000,2.000000,2.000000,5.910000,23.945250,0.148000,2.500000,100.600000,402.500000,13.700000,57.400000
50%,2025-12-28 00:00:00,5.000000,7.000000,29290.000000,4.000000,3.000000,4.000000,4.000000,6.290000,44.190500,0.478000,7.900000,100.600000,710.550000,14.300000,104.900000
75%,2026-01-17 00:00:00,7.000000,8.000000,49000.000000,6.000000,5.000000,6.000000,6.000000,6.700000,67.623750,0.954000,15.300000,100.900000,1109.800000,15.100000,156.300000
max,2026-02-05 00:00:00,10.000000,9.000000,100000.000000,11.000000,8.500000,11.000000,11.000000,16.770000,126.792000,23.291000,324.400000,201.100000,1827.700000,32.300000,276.200000
std,NaN,2.684794,1.153684,21194.516576,2.337252,2.161394,2.329619,2.329619,0.624572,27.589237,0.947022,13.282533,2.162208,436.930423,1.311357,61.925109


In [217]:
# Check the values in grade
laurel_raw["grade"].value_counts()

grade
    14642
Name: count, dtype: int64

In [218]:
# Check the values in about_distance_indicator
laurel_raw["about_distance_indicator"].value_counts()

about_distance_indicator
    14642
Name: count, dtype: int64

Here, weirdly that we don't see any missing data from any columns, which is weird. From the data snippet and by looking at the actual data, we expect that `grade` and `about_distance_indicator` are just empty columns. Thus, we will drop these later before doing any further cleaning, since they don't provide any additional information to our dataset.

Also, note that there are some inconsistency in the column name. We will change them all to lower_snake_case later.

### b. The Traditional Non-GPS Starter Dataset

In [219]:
# First 5 rows of pps_raw
pps_raw.head()

,gps_date?,registration_number,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,distance_unit,about_distance_indicator,Distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,Purse
0,,22017307,A Cozy Thing,MED,2025-10-10,4,USA,AOC,,5.0,F,,5F,T,11,9,9,0,0,5,3,810,780,0,0,750,325,100,200,0,0,10,100,144,11,53550
1,,22017307,A Cozy Thing,CT,2025-10-30,1,USA,CLM,,6.5,F,,6 1/2F,D,4,6,5,0,0,4,3,770,700,0,0,650,460,300,200,0,0,300,50,27,7,21500
2,X,22017307,A Cozy Thing,LRL,2025-11-28,2,USA,SOC,,8.0,F,,1M,D,4,5,7,6,0,5,4,170,420,570,0,810,575,100,200,150,0,400,500,208,8,25740
3,X,21003092,A P M Notion,LRL,2025-10-25,7,USA,CLM,,6.0,F,,6F,D,6,6,4,0,0,4,4,520,410,0,0,550,975,150,100,0,0,250,50,84,8,25700
4,X,21003092,A P M Notion,LRL,2025-11-15,10,USA,CLM,,5.5,F,,5 1/2F,D,4,6,6,0,0,7,6,560,800,0,0,1100,1525,250,150,0,0,0,50,181,7,23220


In [220]:
# Basic information of columns in pps_raw
pps_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2897 entries, 0 to 2896
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   gps_date?                    2897 non-null   object        
 1   registration_number          2897 non-null   object        
 2   horse_name                   2897 non-null   object        
 3   pp_track                     2897 non-null   object        
 4   pp_race_date                 2897 non-null   datetime64[ns]
 5   pp_race_number               2897 non-null   int64         
 6   pp_country                   2897 non-null   object        
 7   race_type                    2897 non-null   object        
 8   grade                        2897 non-null   object        
 9   distance_id                  2897 non-null   float64       
 10  distance_unit                2897 non-null   object        
 11  about_distance_indicator     2897 non-null 

In [221]:
# Basic statistical summary
pps_raw.describe()

,pp_race_date,pp_race_number,distance_id,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,Purse
count,2897,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2897.000000,2.897000e+03
mean,2025-10-21 09:54:59.275112192,5.265102,6.866130,4.454953,4.417673,4.261995,1.478771,0.009320,4.177425,4.106662,417.623749,446.653780,156.395582,4.321367,551.664826,753.806006,139.180532,144.292717,65.152917,3.486020,214.602347,308.291681,139.876769,7.847428,4.515887e+04
min,2024-06-01 00:00:00,1.000000,4.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,7.900000e+03
25%,2025-10-09 00:00:00,3.000000,6.000000,2.000000,2.000000,2.000000,0.000000,0.000000,2.000000,2.000000,100.000000,100.000000,0.000000,0.000000,150.000000,175.000000,10.000000,10.000000,0.000000,0.000000,10.000000,50.000000,30.000000,7.000000,2.552000e+04
50%,2025-11-08 00:00:00,5.000000,6.500000,4.000000,4.000000,4.000000,0.000000,0.000000,4.000000,4.000000,310.000000,340.000000,0.000000,0.000000,450.000000,550.000000,100.000000,100.000000,0.000000,0.000000,150.000000,150.000000,65.000000,8.000000,3.200000e+04
75%,2025-11-29 00:00:00,7.000000,8.000000,6.000000,6.000000,6.000000,2.000000,0.000000,6.000000,6.000000,610.000000,660.000000,150.000000,0.000000,810.000000,1050.000000,200.000000,200.000000,50.000000,0.000000,250.000000,375.000000,159.000000,9.000000,5.000000e+04
max,2026-01-23 00:00:00,15.000000,20.000000,13.000000,13.000000,12.000000,12.000000,9.000000,12.000000,12.000000,9999.000000,9999.000000,9999.000000,9999.000000,9999.000000,10450.000000,9999.000000,9999.000000,9999.000000,9999.000000,9999.000000,9999.000000,1613.000000,14.000000,2.000000e+06
std,NaN,2.763615,1.211301,2.504855,2.490355,2.475023,2.438678,0.256644,2.425464,2.410128,521.781838,546.289172,405.185060,188.753272,672.570361,923.407127,397.237821,399.100136,304.247679,185.781532,540.965535,667.528939,193.148805,1.805700,6.883020e+04


In [222]:
# Check the values in `gps_date?`
pps_raw["gps_date?"].value_counts()

gps_date?
X    2276
      621
Name: count, dtype: int64

In [223]:
# Check the values in `grade`
pps_raw["grade"].value_counts()

grade
     2879
3      11
2       4
1       3
Name: count, dtype: int64

In [224]:
# Check the values in `about_distance_indicator`
pps_raw["about_distance_indicator"].value_counts()

about_distance_indicator
     2861
A      36
Name: count, dtype: int64

We see quite the same problem as in the `laurel_raw` dataframe. A lot of `NaN` values are recorded weirdly, thus do not count as missing values. We will have to fix this and the naming convention in the next section. Also, we need to rename the `gps_date?` column, given its current name is misleading.

### c. The GPS Starter Dataset

In [225]:
# First 5 rows of gps_pp_raw
gps_pp_raw.head()

,horse_name,registration_number,pp_track,pp_race_date,pp_race_number,race_type,grade,distance_id,distance_unit,about_distance_indicator,published_value,surface,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,Purse,field_size
0,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,F,,1M,D,4,0.0,4,4,7.01,100.562,1.022,13.3,100.6,1612.5,15.0,224.8,25740,8
1,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,F,,1M,D,4,0.5,4,4,6.83,93.551,1.151,15.9,100.6,1511.9,14.7,209.8,25740,8
2,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,F,,1M,D,4,1.0,4,4,6.72,86.728,1.198,17.2,100.6,1411.3,14.4,195.1,25740,8
3,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,F,,1M,D,4,1.5,5,4,6.69,80.008,1.047,15.4,100.6,1310.7,14.4,180.7,25740,8
4,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,F,,1M,D,4,2.0,6,4,6.68,73.313,0.863,12.9,101.1,1210.1,14.2,166.3,25740,8


In [226]:
# Basic information of columns in gps_pp_raw
gps_pp_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31174 entries, 0 to 31173
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   horse_name                31174 non-null  object        
 1   registration_number       31174 non-null  object        
 2   pp_track                  31174 non-null  object        
 3   pp_race_date              31174 non-null  datetime64[ns]
 4   pp_race_number            31174 non-null  int64         
 5   race_type                 31174 non-null  object        
 6   grade                     31174 non-null  object        
 7   distance_id               31174 non-null  float64       
 8   distance_unit             31174 non-null  object        
 9   about_distance_indicator  31174 non-null  object        
 10  published_value           31174 non-null  object        
 11  surface                   31174 non-null  object        
 12  post_position     

In [227]:
# Basic statistical summary
gps_pp_raw.describe()

,pp_race_date,pp_race_number,distance_id,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,Purse,field_size
count,31174,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,31174.000000,3.117400e+04,31174.000000
mean,2025-10-25 14:41:48.705973248,5.236704,7.060101,4.520722,6.284158,4.387214,4.237538,6.216121,46.127017,0.660899,10.527292,101.044755,763.708902,14.329977,107.708902,4.737561e+04,7.989928
min,2024-06-01 00:00:00,1.000000,4.500000,1.000000,0.000000,1.000000,1.000000,1.700000,1.696000,0.000000,0.000000,36.600000,94.800000,0.000000,0.000000,1.565500e+04,3.000000
25%,2025-10-17 00:00:00,3.000000,6.000000,2.000000,1.500000,2.000000,2.000000,5.810000,23.567250,0.156000,2.600000,100.600000,402.500000,13.600000,56.900000,2.587000e+04,7.000000
50%,2025-11-14 00:00:00,5.000000,7.000000,4.000000,3.000000,4.000000,4.000000,6.160000,43.123000,0.480000,8.000000,100.600000,709.250000,14.100000,102.800000,3.290000e+04,8.000000
75%,2025-12-04 00:00:00,7.000000,8.000000,6.000000,5.000000,6.000000,6.000000,6.550000,66.189000,0.921750,15.100000,101.000000,1109.300000,14.900000,153.400000,5.161500e+04,9.000000
max,2026-01-23 00:00:00,13.000000,9.500000,13.000000,6514.000000,13.000000,12.000000,12.360000,126.792000,17.962000,215.700000,201.100000,1918.200000,32.300000,276.200000,2.000000e+06,14.000000
std,NaN,2.798281,1.171368,2.537576,116.907339,2.475783,2.409082,0.558545,26.775180,0.738268,10.909746,1.710389,433.535010,1.221965,60.573342,7.935796e+04,1.777045


In [228]:
# Check the values in `grade`
gps_pp_raw["grade"].value_counts()

grade
     30956
3       98
2       64
1       56
Name: count, dtype: int64

In [229]:
# Check the values in `about_distance_indicator`
gps_pp_raw["about_distance_indicator"].value_counts()

about_distance_indicator
     31133
A       41
Name: count, dtype: int64

We see the very same problem as above. We will fix that in the following section.

## 2. Cleaning Helpers

These helpers keep the rest of the notebook readable.

A few important design choices:

- **IDs stay as strings**.  
  `registration_number` is not purely numeric in all rows, so we should not coerce it to an integer.

- **Blank strings become missing values**.  
  The workbook uses blank cells for many concepts. We first convert true blanks to missing values, then handle **structural missingness** explicitly.

- **Distance gets a canonical numeric form**.  
  Even though the published race labels look like `1M`, `6F`, or `1M 40Y`, the database distance is consistently provided in furlongs through `distance_id`. We keep both:
  - `distance_furlongs` for computation
  - `distance_label` for human-readable reporting

In [230]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase column names and replace punctuation/spaces with underscores"""
    out = df.copy()
    out.columns = (
        pd.Index(out.columns)
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        .str.strip("_")
    )
    return out

In [231]:
def strip_strings_and_blank_to_na(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace and convert blank-like strings to missing values"""
    out = df.copy()
    obj_cols = out.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        s = out[col].astype("string").str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA, "na": pd.NA})
        out[col] = s
    return out

In [232]:
def parse_dates(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce")
    return out

In [233]:
def coerce_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

In [234]:
def safe_string_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()

In [235]:
def add_distance_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Create canonical distance columns that are consistent across all three tables"""
    out = df.copy()

    if "distance_id" in out.columns:
        out["distance_furlongs"] = pd.to_numeric(out["distance_id"], errors="coerce")
        out["distance_meters_expected"] = out["distance_furlongs"] * FURLONG_TO_METERS

    if "distance" in out.columns:
        out["distance_label"] = out["distance"].astype("string").str.strip()
    elif "published_value" in out.columns:
        out["distance_label"] = out["published_value"].astype("string").str.strip()

    if "distance_label" in out.columns:
        out["is_nonstandard_distance"] = out["distance_label"].str.contains("Y", na=False).astype("Int64")

    return out

In [236]:
def handle_structural_missingness(
    df: pd.DataFrame, gps_flag_col: str | None = None
) -> pd.DataFrame:
    """Handle columns where blanks carry meaning rather than ordinary missingness"""
    out = df.copy()

    if "about_distance_indicator" in out.columns:
        out["about_distance_indicator"] = out["about_distance_indicator"].astype("string").str.strip()
        out["is_about_distance"] = (out["about_distance_indicator"].fillna("") == "A").astype("Int64")

    if "grade" in out.columns:
        out["grade"] = pd.to_numeric(out["grade"], errors="coerce")
        out["is_graded_race"] = out["grade"].notna().astype("Int64")

    if gps_flag_col is not None and gps_flag_col in out.columns:
        out[gps_flag_col] = out[gps_flag_col].astype("string").str.strip()
        out["has_gps_data"] = (out[gps_flag_col].fillna("") == "X").astype("Int64")

    return out

In [237]:
def add_keys(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Create stable string keys but do not merge the tables yet"""
    out = df.copy()

    if table_name == "laurel":
        out["race_key"] = (
            safe_string_id(out["track_id"])
            + "|"
            + out["race_date"].dt.strftime("%Y-%m-%d")
            + "|"
            + safe_string_id(out["race_number"])
        )
        out["horse_race_key"] = out["race_key"] + "|" + safe_string_id(out["registration_number"])

    elif table_name in {"pps", "gps_pp"}:
        out["pp_race_key"] = (
            safe_string_id(out["registration_number"])
            + "|"
            + safe_string_id(out["pp_track"])
            + "|"
            + out["pp_race_date"].dt.strftime("%Y-%m-%d")
            + "|"
            + safe_string_id(out["pp_race_number"])
        )

    return out

In [238]:
def missing_summary(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Return the columns with the most missing values"""
    out = (
        df.isna()
        .sum()
        .rename("n_missing")
        .reset_index()
        .rename(columns={"index": "column"})
        .query("n_missing > 0")
        .sort_values(["n_missing", "column"], ascending=[False, True])
        .head(top_n)
    )
    return out


In [239]:
def dedupe_exact_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop exact duplicated rows and report how many were removed"""
    n_dupes = int(df.duplicated().sum())
    out = df.drop_duplicates().reset_index(drop=True)
    return out, n_dupes

## 3. Clean the Laurel Park GPS races table

This is the current Laurel Park target-race table.

Important facts about its structure:

- one horse appears multiple times within a race because GPS is recorded at multiple **gates**
- the highest gate is the earliest collection point
- gate `0` is the final point

For preprocessing, we want **two versions** of this table:

1. `laurel_gate_clean`  
   still gate-level, but cleaned and keyed

2. `laurel_horse_race_flat`  
   one row per horse-race, with raw gate columns pivoted wide

In [240]:

def clean_gate_level_table(df: pd.DataFrame) -> pd.DataFrame:
    out = standardize_columns(df)
    out = strip_strings_and_blank_to_na(out)
    out = parse_dates(out, ["race_date", "pp_race_date"])

    numeric_cols = [
        "race_number",
        "pp_race_number",
        "distance_id",
        "grade",
        "purse",
        "post_position",
        "gate",
        "position",
        "official_position",
        "sectional_time",
        "running_time",
        "time_behind",
        "distance_behind",
        "distance_ran",
        "cumulative_distance_ran",
        "strides",
        "cumulative_strides",
        "field_size",
    ]
    out = coerce_numeric(out, numeric_cols)
    out = handle_structural_missingness(out)
    out = add_distance_fields(out)
    return out

In [241]:
laurel_gate_clean = clean_gate_level_table(laurel_raw)
laurel_gate_clean = add_keys(laurel_gate_clean, "laurel")

# sort so that within each horse-race we move from earliest gate down to gate 0
laurel_gate_clean = laurel_gate_clean.sort_values(
    ["race_date", "race_number", "registration_number", "gate"],
    ascending=[True, True, True, False],
).reset_index(drop=True)

display(laurel_gate_clean.head(10))

,track_id,race_date,race_number,distance_id,distance_unit,about_distance_indicator,distance,surface,race_type,grade,purse,registration_number,horse_name,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,is_about_distance,is_graded_race,distance_furlongs,distance_meters_expected,distance_label,is_nonstandard_distance,race_key,horse_race_key
0,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,6.5,8,7,7.49,7.485,0.953,16.3,100.6,100.6,18.8,18.8,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
1,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,6.0,8,7,5.51,12.994,0.861,15.6,100.6,201.2,13.6,32.4,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
2,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,5.5,7,7,5.55,18.543,0.714,12.8,100.6,301.8,13.6,46.0,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
3,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,5.0,7,7,5.64,24.184,0.644,11.3,100.6,402.4,13.9,59.9,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
4,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,4.5,6,7,5.73,29.910,0.622,10.7,100.6,503.0,13.9,73.8,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
5,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,4.0,6,7,5.90,35.806,0.671,11.2,100.6,603.6,14.2,88.0,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
6,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,3.5,6,7,5.95,41.749,0.819,13.3,100.6,704.2,14.4,102.4,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
7,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,3.0,7,7,6.21,47.956,0.926,14.4,100.7,804.9,15.0,117.4,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
8,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,2.5,5,7,6.61,54.572,0.970,14.7,102.5,907.4,16.0,133.4,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714
9,LRL,2025-12-12,1,7.0,F,<NA>,7F,D,SOC,<NA>,27900,19002714,Mischief Motion,4,2.0,5,7,6.98,61.555,1.098,16.3,104.6,1012.0,16.4,149.8,0,0,7.0,1408.176,7F,0,LRL|2025-12-12|1,LRL|2025-12-12|1|19002714


In [242]:
laurel_clean_inventory = pd.DataFrame(
    {
        "rows": [len(laurel_gate_clean)],
        "horse_races": [laurel_gate_clean["horse_race_key"].nunique()],
        "races": [laurel_gate_clean["race_key"].nunique()],
        "min_gate": [laurel_gate_clean["gate"].min()],
        "max_gate": [laurel_gate_clean["gate"].max()],
    },
    index=["laurel_gate_clean"],
)
display(laurel_clean_inventory)

,rows,horse_races,races,min_gate,max_gate
laurel_gate_clean,14642,1054,148,0.0,8.5


In [243]:
display(missing_summary(laurel_gate_clean))

,column,n_missing
5,about_distance_indicator,14642
9,grade,14642


The cleaned Laurel table now has the structure we want for the target races:

* It still has **14,642 rows**, but those rows collapses to only **1,054 unique horse-race observations** across **148 races**, confirming that this sheet is fundamentally a repeated gate-level table

* After converting blank strings to missing values, both `grade` and `about_distance_indicator` are 100%, so these are structural empty columns rather than useful features. Thus, we could do a step further here and drop these two columns.

* The meaningful preprocessing decision here is therefore to preserve keys, gate ordering, and distance fields, not to spend effort imputing those empty columns.

In [244]:
# Drop grade and about_distance_indicator columns
laurel_gate_clean.drop(columns=['grade', 'about_distance_indicator'], inplace=True)
display(missing_summary(laurel_gate_clean))

,column,n_missing


### Flatten Laurel to one row per horse-race

This step is still **preprocessing**, not feature engineering.

We are not summarizing gates into pace features yet. Instead, we widen the raw gate columns into columns like:

- `gate_7p5_position`
- `gate_7p5_sectional_time`
- `gate_0_running_time`

That gives Notebook 2 a clean horse-race table while preserving the original gate-level information.

In [245]:

def flatten_laurel_gate_table(laurel_df: pd.DataFrame) -> pd.DataFrame:
    static_cols = [
        "horse_race_key",
        "race_key",
        "track_id",
        "race_date",
        "race_number",
        "distance_id",
        "distance_furlongs",
        "distance_meters_expected",
        "distance_unit",
        "distance_label",
        "is_nonstandard_distance",
        "about_distance_indicator",
        "is_about_distance",
        "surface",
        "race_type",
        "grade",
        "is_graded_race",
        "purse",
        "registration_number",
        "horse_name",
        "post_position",
        "official_position",
    ]
    static_cols = [c for c in static_cols if c in laurel_df.columns]

    static_part = (
        laurel_df[static_cols]
        .drop_duplicates(subset="horse_race_key")
        .sort_values(["race_date", "race_number", "horse_name"])
    )

    # field size belongs to the race, not the horse
    field_size = (
        laurel_df[["race_key", "registration_number"]]
        .drop_duplicates()
        .groupby("race_key")
        .size()
        .rename("field_size")
        .reset_index()
    )

    gate_meta = (
        laurel_df.groupby("horse_race_key")
        .agg(
            num_gate_rows=("gate", "size"),
            gate_min=("gate", "min"),
            gate_max=("gate", "max"),
        )
        .reset_index()
    )

    value_cols = [
        "position",
        "sectional_time",
        "running_time",
        "time_behind",
        "distance_behind",
        "distance_ran",
        "cumulative_distance_ran",
        "strides",
        "cumulative_strides",
    ]

    tmp = laurel_df[["horse_race_key", "gate"] + value_cols].copy()
    tmp["gate_label"] = tmp["gate"].map(lambda x: f"gate_{str(x).replace('.', 'p')}")

    wide_parts = []
    for value_col in value_cols:
        part = tmp.pivot(index="horse_race_key", columns="gate_label", values=value_col)
        part = part.rename(columns={c: f"{c}_{value_col}" for c in part.columns})
        wide_parts.append(part)

    wide_gate_values = pd.concat(wide_parts, axis=1).reset_index()

    out = (
        static_part
        .merge(gate_meta, on="horse_race_key", how="left")
        .merge(field_size, on="race_key", how="left")
        .merge(wide_gate_values, on="horse_race_key", how="left")
        .sort_values(["race_date", "race_number", "official_position", "horse_name"])
        .reset_index(drop=True)
    )

    return out

In [246]:
laurel_horse_race_flat = flatten_laurel_gate_table(laurel_gate_clean)

display(
    pd.DataFrame(
        {
            "rows": [len(laurel_horse_race_flat)],
            "columns": [laurel_horse_race_flat.shape[1]],
            "unique_horse_race_keys": [laurel_horse_race_flat["horse_race_key"].nunique()],
            "unique_races": [laurel_horse_race_flat["race_key"].nunique()],
        },
        index=["laurel_horse_race_flat"],
    )
)

,rows,columns,unique_horse_race_keys,unique_races
laurel_horse_race_flat,1054,186,1054,148


In [247]:
display(laurel_horse_race_flat.head(10))

,horse_race_key,race_key,track_id,race_date,race_number,distance_id,distance_furlongs,distance_meters_expected,distance_unit,distance_label,is_nonstandard_distance,is_about_distance,surface,race_type,is_graded_race,purse,registration_number,horse_name,post_position,official_position,num_gate_rows,gate_min,gate_max,field_size,gate_0p0_position,gate_0p5_position,gate_1p0_position,gate_1p5_position,gate_2p0_position,gate_2p5_position,gate_3p0_position,gate_3p5_position,gate_4p0_position,gate_4p5_position,gate_5p0_position,gate_5p5_position,gate_6p0_position,gate_6p5_position,gate_7p0_position,gate_7p5_position,gate_8p0_position,gate_8p5_position,gate_0p0_sectional_time,gate_0p5_sectional_time,gate_1p0_sectional_time,gate_1p5_sectional_time,gate_2p0_sectional_time,gate_2p5_sectional_time,gate_3p0_sectional_time,gate_3p5_sectional_time,gate_4p0_sectional_time,gate_4p5_sectional_time,gate_5p0_sectional_time,gate_5p5_sectional_time,gate_6p0_sectional_time,gate_6p5_sectional_time,gate_7p0_sectional_time,gate_7p5_sectional_time,gate_8p0_sectional_time,gate_8p5_sectional_time,gate_0p0_running_time,gate_0p5_running_time,gate_1p0_running_time,gate_1p5_running_time,gate_2p0_running_time,gate_2p5_running_time,gate_3p0_running_time,gate_3p5_running_time,gate_4p0_running_time,gate_4p5_running_time,gate_5p0_running_time,gate_5p5_running_time,gate_6p0_running_time,gate_6p5_running_time,gate_7p0_running_time,gate_7p5_running_time,gate_8p0_running_time,gate_8p5_running_time,gate_0p0_time_behind,gate_0p5_time_behind,gate_1p0_time_behind,gate_1p5_time_behind,gate_2p0_time_behind,gate_2p5_time_behind,gate_3p0_time_behind,gate_3p5_time_behind,gate_4p0_time_behind,gate_4p5_time_behind,gate_5p0_time_behind,gate_5p5_time_behind,gate_6p0_time_behind,gate_6p5_time_behind,gate_7p0_time_behind,gate_7p5_time_behind,gate_8p0_time_behind,gate_8p5_time_behind,gate_0p0_distance_behind,gate_0p5_distance_behind,gate_1p0_distance_behind,gate_1p5_distance_behind,gate_2p0_distance_behind,gate_2p5_distance_behind,gate_3p0_distance_behind,gate_3p5_distance_behind,gate_4p0_distance_behind,gate_4p5_distance_behind,gate_5p0_distance_behind,gate_5p5_distance_behind,gate_6p0_distance_behind,gate_6p5_distance_behind,gate_7p0_distance_behind,gate_7p5_distance_behind,gate_8p0_distance_behind,gate_8p5_distance_behind,gate_0p0_distance_ran,gate_0p5_distance_ran,gate_1p0_distance_ran,gate_1p5_distance_ran,gate_2p0_distance_ran,gate_2p5_distance_ran,gate_3p0_distance_ran,gate_3p5_distance_ran,gate_4p0_distance_ran,gate_4p5_distance_ran,gate_5p0_distance_ran,gate_5p5_distance_ran,gate_6p0_distance_ran,gate_6p5_distance_ran,gate_7p0_distance_ran,gate_7p5_distance_ran,gate_8p0_distance_ran,gate_8p5_distance_ran,gate_0p0_cumulative_distance_ran,gate_0p5_cumulative_distance_ran,gate_1p0_cumulative_distance_ran,gate_1p5_cumulative_distance_ran,gate_2p0_cumulative_distance_ran,gate_2p5_cumulative_distance_ran,gate_3p0_cumulative_distance_ran,gate_3p5_cumulative_distance_ran,gate_4p0_cumulative_distance_ran,gate_4p5_cumulative_distance_ran,gate_5p0_cumulative_distance_ran,gate_5p5_cumulative_distance_ran,gate_6p0_cumulative_distance_ran,gate_6p5_cumulative_distance_ran,gate_7p0_cumulative_distance_ran,gate_7p5_cumulative_distance_ran,gate_8p0_cumulative_distance_ran,gate_8p5_cumulative_distance_ran,gate_0p0_strides,gate_0p5_strides,gate_1p0_strides,gate_1p5_strides,gate_2p0_strides,gate_2p5_strides,gate_3p0_strides,gate_3p5_strides,gate_4p0_strides,gate_4p5_strides,gate_5p0_strides,gate_5p5_strides,gate_6p0_strides,gate_6p5_strides,gate_7p0_strides,gate_7p5_strides,gate_8p0_strides,gate_8p5_strides,gate_0p0_cumulative_strides,gate_0p5_cumulative_strides,gate_1p0_cumulative_strides,gate_1p5_cumulative_strides,gate_2p0_cumulative_strides,gate_2p5_cumulative_strides,gate_3p0_cumulative_strides,gate_3p5_cumulative_strides,gate_4p0_cumulative_strides,gate_4p5_cumulative_strides,gate_5p0_cumulative_strides,gate_5p5_cumulative_strides,gate_6p0_cumulative_strides,gate_6p5_cumulative_strides,gate_

Flattening the Laurel table gives us a modeling-friendly target table **without losing horse-race coverage**:

* The flattened output has **1,054 rows**, exactly matching the number of unique `horse_race_key` values

* That means each target horse-race is preserved once and only once

* We only keep raw gate measurements in widened form, which is useful because Notebook 2 can engineer pace and trajectory features later without needing to reconstruct the original gate structure

## 4. Clean the traditional PP table

This is the historical **non-GPS** table.

A few special cases matter here:

- the workbook header `gps_date?` is functionally a **GPS-availability flag**, not a date field  
  so we rename it to `gps_data_flag`
- point-of-call columns need careful cleaning
- exact duplicated rows appear in this sheet and can be dropped safely at preprocessing time

In [248]:

def clean_pps_table(df: pd.DataFrame) -> pd.DataFrame:
    out = standardize_columns(df)
    out = strip_strings_and_blank_to_na(out)

    # The raw header becomes `gps_date` after standardization, but it acts as a flag.
    if "gps_date" in out.columns:
        out = out.rename(columns={"gps_date": "gps_data_flag"})

    out = parse_dates(out, ["pp_race_date"])

    numeric_cols = [
        "pp_race_number",
        "distance_id",
        "grade",
        "post_position",
        "official_position",
        "post_time_odds",
        "field_size",
        "purse",
        "position_at_point_of_call_1",
        "position_at_point_of_call_2",
        "position_at_point_of_call_3",
        "position_at_point_of_call_4",
        "position_at_point_of_call_5",
        "length_behind_at_poc_1",
        "length_behind_at_poc_2",
        "length_behind_at_poc_3",
        "length_behind_at_poc_4",
        "length_behind_at_poc_5",
        "length_behind_at_finish",
        "length_ahead_at_poc_1",
        "length_ahead_at_poc_2",
        "length_ahead_at_poc_3",
        "length_ahead_at_poc_4",
        "length_ahead_at_poc_5",
        "length_ahead_at_finish",
    ]
    out = coerce_numeric(out, numeric_cols)

    out = handle_structural_missingness(out, gps_flag_col="gps_data_flag")
    out = add_distance_fields(out)
    out = add_keys(out, "pps")

    # Point-of-call positions:
    # 0 is not a valid racing position, so we treat it as "no such call for this distance".
    poc_position_cols = [c for c in out.columns if re.fullmatch(r"position_at_point_of_call_\d+", c)]
    for col in poc_position_cols:
        out.loc[out[col] == 0, col] = np.nan

    # Length columns:
    # 9999 is the explicit sentinel for "did not finish / unavailable margin".
    poc_length_cols = [
        c
        for c in out.columns
        if re.fullmatch(r"length_(behind|ahead)_at_(poc_\d+|finish)", c)
    ]
    for col in poc_length_cols:
        out.loc[out[col] == 9999, col] = np.nan

    out, pps_exact_dupes_removed = dedupe_exact_rows(out)
    out.attrs["exact_duplicates_removed"] = pps_exact_dupes_removed
    return out


pps_clean = clean_pps_table(pps_raw)
pps_clean = pps_clean.sort_values(
    ["registration_number", "pp_race_date", "pp_race_number"],
    ascending=[True, False, False],
).reset_index(drop=True)

display(pps_clean.head(10))

,gps_data_flag,registration_number,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,distance_unit,about_distance_indicator,distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,purse,is_about_distance,is_graded_race,has_gps_data,distance_furlongs,distance_meters_expected,distance_label,is_nonstandard_distance,pp_race_key
0,X,15000089,Bode's Maker,LRL,2025-11-22,2,USA,SOC,<NA>,8.00,F,<NA>,1M,T,7,10.0,8.0,7.0,NaN,9.0,4,1080.0,810.0,410.0,0.0,520.0,200.0,500.0,200.0,100.0,0.0,800.0,50.0,50,11,26520,0,0,1,8.00,1609.34400,1M,0,15000089|LRL|2025-11-22|2
1,X,15000089,Bode's Maker,LRL,2025-11-01,6,USA,SOC,<NA>,8.50,F,<NA>,1 1/16M,T,6,12.0,9.0,7.0,NaN,4.0,2,920.0,730.0,560.0,0.0,360.0,100.0,0.0,50.0,10.0,0.0,50.0,5.0,39,12,27020,0,0,1,8.50,1709.92800,1 1/16M,0,15000089|LRL|2025-11-01|6
2,X,15000089,Bode's Maker,LRL,2025-10-04,9,USA,SOC,<NA>,8.00,F,<NA>,1M,T,9,7.0,7.0,7.0,NaN,3.0,1,1000.0,1150.0,720.0,0.0,300.0,0.0,250.0,200.0,10.0,0.0,50.0,100.0,36,9,25520,0,0,1,8.00,1609.34400,1M,0,15000089|LRL|2025-10-04|9
3,X,16009053,Tiberius Mercurius,AQU,2025-12-31,8,USA,CLM,<NA>,6.50,F,<NA>,6 1/2F,D,4,11.0,11.0,NaN,NaN,11.0,10,800.0,1140.0,0.0,0.0,1040.0,700.0,0.0,0.0,0.0,0.0,0.0,450.0,404,11,28000,0,0,1,6.50,1307.59200,6 1/2F,0,16009053|AQU|2025-12-31|8
4,X,16009053,Tiberius Mercurius,AQU,2025-12-10,6,USA,CLM,<NA>,8.00,F,<NA>,1M,D,1,4.0,2.0,2.0,NaN,2.0,4,120.0,200.0,50.0,0.0,350.0,725.0,100.0,10.0,150.0,0.0,50.0,75.0,50,7,28000,0,0,1,8.00,1609.34400,1M,0,16009053|AQU|2025-12-10|6
5,X,16009053,Tiberius Mercurius,BAQ,2025-09-18,6,USA,CLM,<NA>,9.00,F,<NA>,1 1/8M,D,7,8.0,8.0,8.0,NaN,7.0,7,1550.0,1720.0,1360.0,0.0,1200.0,1605.0,0.0,0.0,0.0,0.0,200.0,150.0,299,8,32000,0,0,1,9.00,1810.51200,1 1/8M,0,16009053|BAQ|2025-09-18|6
6,X,16011638,Armando R,LRL,2025-11-29,5,USA,SOC,<NA>,8.50,F,<NA>,1 1/16M,D,3,7.0,8.0,7.0,NaN,6.0,1,660.0,880.0,860.0,0.0,420.0,0.0,50.0,100.0,150.0,0.0,200.0,50.0,18,9,31370,0,0,1,8.50,1709.92800,1 1/16M,0,16011638|LRL|2025-11-29|5
7,X,16011638,Armando R,LRL,2025-10-18,2,USA,SOC,<NA>,9.00,F,<NA>,1 1/8M,D,4,6.0,6.0,6.0,NaN,2.0,1,900.0,850.0,750.0,0.0,250.0,0.0,0.0,0.0,0.0,0.0,100.0,125.0,23,6,29435,0,0,1,9.00,1810.51200,1 1/8M,0,16011638|LRL|2025-10-18|2
8,X,16011638,Armando R,LRL,2025-09-21,4,USA,SOC,<NA>,8.50,F,<NA>,1 1/16M,D,1,6.0,7.0,7.0,NaN,2.0,1,1160.0,920.0,720.0,0.0,150.0,0.0,150.0,0.0,0.0,0.0,10.0,225.0,19,7,30370,0,0,1,8.50,1709.92800,1 1/16M,0,16011638|LRL|2025-09-21|4
9,<NA>,16012148,Bold Endeavor,PRX,2025-11-17,10,USA,CLM,<NA>,8.32,F,<NA>,1M 70Y,D,5,5.0,5.0,5.0,NaN,2.0,1,260.0,130.0,210.0,0.0,150.0,0.0,100.0,100.0,400.0,0.0,150.0,225.0,23,9,26000,0,0,0,8.32,1673.71776,1M 70Y,1,16012148|PRX|2025-11-17|10


In [249]:

pps_inventory = pd.DataFrame(
    {
        "rows": [len(pps_clean)],
        "unique_horses": [pps_clean["registration_number"].nunique()],
        "unique_pp_races": [pps_clean["pp_race_key"].nunique()],
        "exact_duplicate_rows_removed": [pps_clean.attrs.get("exact_duplicates_removed", 0)],
        "gps_flagged_pp_races": [pps_clean.loc[pps_clean["has_gps_data"] == 1, "pp_race_key"].nunique()],
    },
    index=["pps_clean"],
)
display(pps_inventory)

,rows,unique_horses,unique_pp_races,exact_duplicate_rows_removed,gps_flagged_pp_races
pps_clean,2293,700,2293,604,1744


In [250]:

# Quick QA: how many rows still use each point-of-call position column after cleaning?
poc_position_cols = [c for c in pps_clean.columns if c.startswith("position_at_point_of_call_")]
poc_presence = pd.DataFrame(
    {
        "column": poc_position_cols,
        "non_missing_after_cleaning": [pps_clean[c].notna().sum() for c in poc_position_cols],
    }
).sort_values("column")
display(poc_presence)

# Quick QA: lengths are kept as coded horse-length units, but 9999 should be gone.
poc_length_cols = [c for c in pps_clean.columns if "length_" in c]
length_sentinel_check = pd.DataFrame(
    {
        "column": poc_length_cols,
        "remaining_9999_values": [(pps_clean[c] == 9999).sum() for c in poc_length_cols],
    }
).sort_values("column")
display(length_sentinel_check.head(20))

,column,non_missing_after_cleaning
0,position_at_point_of_call_1,2293
1,position_at_point_of_call_2,2264
2,position_at_point_of_call_3,834
3,position_at_point_of_call_4,4
4,position_at_point_of_call_5,2293


,column,remaining_9999_values
11,length_ahead_at_finish,0
6,length_ahead_at_poc_1,0
7,length_ahead_at_poc_2,0
8,length_ahead_at_poc_3,0
9,length_ahead_at_poc_4,0
10,length_ahead_at_poc_5,0
5,length_behind_at_finish,0
0,length_behind_at_poc_1,0
1,length_behind_at_poc_2,0
2,length_behind_at_poc_3,0


In [251]:
display(missing_summary(pps_clean))

,column,n_missing
18,position_at_point_of_call_4,2289
8,grade,2279
11,about_distance_indicator,2260
17,position_at_point_of_call_3,1459
0,gps_data_flag,549
16,position_at_point_of_call_2,29
32,length_ahead_at_finish,8
26,length_behind_at_finish,8
31,length_ahead_at_poc_5,6
25,length_behind_at_poc_5,6


The traditional PP table needed substaintial cleanup before it could be trusted:

* Exact deduplication removed **604 rows** (2897 -> 2293), which is a large enough change that it validates doing preprocessing carefully before modeling

* `grade` and `about_distance_indicator` are mostly missing after cleaning, so they look weak as standalone features

* The `gps_data_flag` field is still useful because it marks **1,744** historical PP entries that are expected to have matching GPS detail

* The point-of-call columns are clearly not uniformly populated: for example, `position_at_point_of_call_4` is almost never used, while earlier calls are much more common. That strongly suggests the point-of-call structure depends on the race distance

## 5. Clean the GPS PP table

This is the historical **GPS** detail table for the subset of PP races that actually had GPS.

It has the same gate-level structure as the Laurel table, but it refers to **past races**, not the current Laurel target races.

In [252]:

def clean_gps_pp_table(df: pd.DataFrame) -> pd.DataFrame:
    out = standardize_columns(df)
    out = strip_strings_and_blank_to_na(out)
    out = parse_dates(out, ["pp_race_date"])

    numeric_cols = [
        "pp_race_number",
        "distance_id",
        "grade",
        "post_position",
        "gate",
        "position",
        "official_position",
        "sectional_time",
        "running_time",
        "time_behind",
        "distance_behind",
        "distance_ran",
        "cumulative_distance_ran",
        "strides",
        "cumulative_strides",
        "purse",
        "field_size",
    ]
    out = coerce_numeric(out, numeric_cols)
    out = handle_structural_missingness(out)
    out = add_distance_fields(out)
    out = add_keys(out, "gps_pp")

    out, gps_exact_dupes_removed = dedupe_exact_rows(out)
    out.attrs["exact_duplicates_removed"] = gps_exact_dupes_removed
    return out


gps_pp_clean = clean_gps_pp_table(gps_pp_raw)
gps_pp_clean = gps_pp_clean.sort_values(
    ["registration_number", "pp_race_date", "pp_race_number", "gate"],
    ascending=[True, False, False, False],
).reset_index(drop=True)

display(gps_pp_clean.head(10))

,horse_name,registration_number,pp_track,pp_race_date,pp_race_number,race_type,grade,distance_id,distance_unit,about_distance_indicator,published_value,surface,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,purse,field_size,is_about_distance,is_graded_race,distance_furlongs,distance_meters_expected,distance_label,is_nonstandard_distance,pp_race_key
0,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,7.5,10,4,7.30,7.304,0.891,15.3,100.8,100.8,16.9,16.9,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
1,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,7.0,10,4,5.99,13.295,1.089,18.6,103.3,204.1,14.3,31.2,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
2,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,6.5,9,4,6.05,19.351,1.144,19.3,101.2,305.3,14.2,45.4,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
3,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,6.0,9,4,5.82,25.175,1.225,20.5,100.7,406.0,13.6,59.0,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
4,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,5.5,9,4,5.89,31.067,1.276,21.6,100.7,506.7,13.8,72.8,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
5,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,5.0,8,4,5.90,36.966,1.267,21.4,100.6,607.3,13.8,86.6,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
6,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,4.5,8,4,5.82,42.782,1.129,19.0,100.6,707.9,13.6,100.2,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
7,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,4.0,7,4,5.91,48.688,0.928,15.4,100.6,808.5,13.7,113.9,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
8,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,3.5,7,4,6.02,54.707,0.688,11.3,100.6,909.1,14.0,127.9,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2
9,Bode's Maker,15000089,LRL,2025-11-22,2,SOC,<NA>,8.0,F,<NA>,1M,T,7,3.0,7,4,6.08,60.784,0.524,8.4,101.1,1010.2,14.0,141.9,26520,11,0,0,8.0,1609.344,1M,0,15000089|LRL|2025-11-22|2


In [253]:

gps_pp_inventory = pd.DataFrame(
    {
        "rows": [len(gps_pp_clean)],
        "unique_horses": [gps_pp_clean["registration_number"].nunique()],
        "unique_pp_races": [gps_pp_clean["pp_race_key"].nunique()],
        "exact_duplicate_rows_removed": [gps_pp_clean.attrs.get("exact_duplicates_removed", 0)],
    },
    index=["gps_pp_clean"],
)
display(gps_pp_inventory)

,rows,unique_horses,unique_pp_races,exact_duplicate_rows_removed
gps_pp_clean,23971,627,1739,7203


In [254]:
display(missing_summary(gps_pp_clean))

,column,n_missing
9,about_distance_indicator,23930
6,grade,23802


The historical GPS PP table preserves much richer within-race detail than the traditional PP sheet:

* Even after removing **7,203** exact duplicate rows, the cleaned table still contains **23,971 rows** linked to **1,739 unique PP keys**

* This confirms that the GPS data is the high-resolution historical source we will later use for GPS-based features

* As in the other tables, `grade` and `about_distance_indicator` behave like structural empty fields and should not drive downstream feature design

## 6. Build a distance-aware point-of-call reference

Equibase's point-of-call structure varies by race distance.  
Rather than hard-coding assumptions too early, we build a **data-driven reference table** from the cleaned PP sheet.

This lookup answers a simple preprocessing question:

> For each `distance_label`, which generic point-of-call columns (`poc_1` through `poc_5`) are actually populated in this workbook?

That makes Notebook 2 much safer, because it can respect the point-of-call structure actually present in the data.

In [255]:

point_of_call_profile_by_distance = (
    pps_clean.groupby(["distance_label", "distance_furlongs"], dropna=False)
    .agg(
        n_pp_rows=("registration_number", "size"),
        n_unique_pp_races=("pp_race_key", "nunique"),
        active_poc_1=("position_at_point_of_call_1", lambda s: s.notna().sum()),
        active_poc_2=("position_at_point_of_call_2", lambda s: s.notna().sum()),
        active_poc_3=("position_at_point_of_call_3", lambda s: s.notna().sum()),
        active_poc_4=("position_at_point_of_call_4", lambda s: s.notna().sum()),
        active_poc_5=("position_at_point_of_call_5", lambda s: s.notna().sum()),
    )
    .reset_index()
    .sort_values(["distance_furlongs", "distance_label"])
)

active_cols = [c for c in point_of_call_profile_by_distance.columns if c.startswith("active_poc_")]
point_of_call_profile_by_distance["num_active_point_of_calls"] = (
    point_of_call_profile_by_distance[active_cols] > 0
).sum(axis=1)

point_of_call_profile_by_distance["active_point_of_call_columns"] = point_of_call_profile_by_distance.apply(
    lambda row: ", ".join([f"poc_{i}" for i in range(1, 6) if row[f"active_poc_{i}"] > 0]),
    axis=1,
)

display(point_of_call_profile_by_distance)

,distance_label,distance_furlongs,n_pp_rows,n_unique_pp_races,active_poc_1,active_poc_2,active_poc_3,active_poc_4,active_poc_5,num_active_point_of_calls,active_point_of_call_columns
11,4F,4.00,4,4,4,0,0,0,4,2,"poc_1, poc_5"
10,4 1/2F,4.50,25,25,25,0,0,0,25,2,"poc_1, poc_5"
13,5F,5.00,25,25,25,25,0,0,25,3,"poc_1, poc_2, poc_5"
12,5 1/2F,5.50,286,286,286,286,0,0,286,3,"poc_1, poc_2, poc_5"
15,6F,6.00,772,772,772,772,0,0,772,3,"poc_1, poc_2, poc_5"
14,6 1/2F,6.50,94,94,94,94,0,0,94,3,"poc_1, poc_2, poc_5"
17,7F,7.00,245,245,245,245,0,0,245,3,"poc_1, poc_2, poc_5"
16,7 1/2F,7.50,8,8,8,8,0,0,8,3,"poc_1, poc_2, poc_5"
5,1M,8.00,406,406,406,406,406,0,406,4,"poc_1, poc_2, poc_3, poc_5"
6,1M 40Y,8.18,1,1,1,1,1,0,1,4,"poc_1, poc_2, poc_3, poc_5"


This table confirms that the Equibase point-of-call structure is **distance-dependent**, not universal.

In practice, that means:

- shorter races may only use a subset of the generic point-of-call columns,
- longer races can activate more intermediate calls,
- and any later feature engineering should respect this lookup rather than assume `poc_1` through `poc_5` mean the same thing everywhere.

Building this reference now makes Notebook 2 safer and more interpretable.

## 7. Coverage checks

These checks do **not** merge the tables.  
They just validate that the workbook is internally consistent enough for the next notebook.

Main question:
- among PP races marked with GPS available, how many have a matching `pp_race_key` in `gps_pp_clean`?

In [256]:

gps_flag_coverage = (
    pps_clean.loc[pps_clean["has_gps_data"] == 1, ["registration_number", "horse_name", "pp_track", "pp_race_date", "pp_race_number", "pp_race_key"]]
    .drop_duplicates()
    .assign(has_matching_gps_pp=lambda d: d["pp_race_key"].isin(set(gps_pp_clean["pp_race_key"])))
)

gps_flag_coverage_summary = (
    gps_flag_coverage["has_matching_gps_pp"]
    .value_counts(dropna=False)
    .rename_axis("has_matching_gps_pp")
    .reset_index(name="count")
)

gps_flag_coverage_issues = gps_flag_coverage.loc[~gps_flag_coverage["has_matching_gps_pp"]].copy()

display(gps_flag_coverage_summary)

,has_matching_gps_pp,count
0,True,1739
1,False,5


In [257]:
display(gps_flag_coverage_issues)

,registration_number,horse_name,pp_track,pp_race_date,pp_race_number,pp_race_key,has_matching_gps_pp
357,20004104,Lucked In,LRL,2025-09-13,9,20004104|LRL|2025-09-13|9,False
431,20011779,Gluckstadt,LRL,2025-11-23,7,20011779|LRL|2025-11-23|7,False
671,21004535,Proud Divi,LRL,2025-11-07,2,21004535|LRL|2025-11-07|2,False
728,21006267,Firstlady Rosalynn,AQU,2025-11-28,3,21006267|AQU|2025-11-28|3,False
1963,23007154,Broadside Salvo,LRL,2025-10-25,9,23007154|LRL|2025-10-25|9,False


The internal coverage check is reassuring:

- Of the **1,744** PP entries flagged as having GPS data, **1,739** successfully match a row in `gps_pp_clean`.
- Only **5** flagged entries do not match.

That is strong enough coverage to move forward confidently, while still saving the mismatch table as a QA artifact in case those five cases reflect workbook inconsistencies or key-format edge cases.

## 8. Save clean outputs

These are the files Notebook 2 should read.

We save:

- `laurel_gate_clean.csv`
- `laurel_horse_race_flat.csv`
- `pps_clean.csv`
- `gps_pp_clean.csv`
- `point_of_call_profile_by_distance.csv`
- `gps_flag_coverage_issues.csv`

In [258]:
laurel_gate_path = OUTPUT_DIR / "laurel_gate_clean.csv"
laurel_flat_path = OUTPUT_DIR / "laurel_horse_race_flat.csv"
pps_path = OUTPUT_DIR / "pps_clean.csv"
gps_pp_path = OUTPUT_DIR / "gps_pp_clean.csv"
poc_profile_path = OUTPUT_DIR / "point_of_call_profile_by_distance.csv"
gps_issues_path = OUTPUT_DIR / "gps_flag_coverage_issues.csv"

laurel_gate_clean.to_csv(laurel_gate_path, index=False)
laurel_horse_race_flat.to_csv(laurel_flat_path, index=False)
pps_clean.to_csv(pps_path, index=False)
gps_pp_clean.to_csv(gps_pp_path, index=False)
point_of_call_profile_by_distance.to_csv(poc_profile_path, index=False)
gps_flag_coverage_issues.to_csv(gps_issues_path, index=False)

saved_files = pd.DataFrame(
    {
        "file": [
            laurel_gate_path.name,
            laurel_flat_path.name,
            pps_path.name,
            gps_pp_path.name,
            poc_profile_path.name,
            gps_issues_path.name,
        ],
        "path": [
            str(laurel_gate_path),
            str(laurel_flat_path),
            str(pps_path),
            str(gps_pp_path),
            str(poc_profile_path),
            str(gps_issues_path),
        ],
    }
)
display(saved_files)

,file,path
0,laurel_gate_clean.csv,../data/processed/laurel_gate_clean.csv
1,laurel_horse_race_flat.csv,../data/processed/laurel_horse_race_flat.csv
2,pps_clean.csv,../data/processed/pps_clean.csv
3,gps_pp_clean.csv,../data/processed/gps_pp_clean.csv
4,point_of_call_profile_by_distance.csv,../data/processed/point_of_call_profile_by_dis...
5,gps_flag_coverage_issues.csv,../data/processed/gps_flag_coverage_issues.csv


## 9. Final checkpoint

At the end of Notebook 1, we now have:

- a cleaned gate-level Laurel table
- a flattened one-row-per-horse-race Laurel table
- a cleaned traditional PP table
- a cleaned historical GPS PP table
- a distance-aware point-of-call lookup
- a small QA table for GPS coverage mismatches

That is enough preprocessing to move cleanly into Notebook 2, where we can do:

- feature engineering
- historical aggregation
- table connections
- model-ready dataset construction